# Imports

In [7]:
import os
import pickle
import random
import typing

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    resnet50, ResNet50_Weights,
    densenet121, DenseNet121_Weights
)
from torch.cuda.amp import autocast

from tqdm import tqdm

# Base Config

In [8]:
class Config:

    SEED = 42
    IMAGE_SIZE = 320
    NUM_CLASSES = 3
    LABELS = [
        "Atelectasis",
        "Effusion",
        "Pneumothorax"
    ]
    MODEL_ARCH = "" 
    BATCH_SIZE = 32
    EPOCHS = 30
    LR = 0.00012638017994009575
    DEVICE = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    NUM_WORKERS = 4 if DEVICE.type == "cuda" else 2
    PIN_MEMORY = DEVICE.type == "cuda"
    DATA_DIR = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
    CSV_PATH = os.path.join(
        DATA_DIR,
        "Data_Entry_2017.csv"
    )
    MODEL_DIR = "models"

os.makedirs(Config.MODEL_DIR, exist_ok=True)

In [9]:
def set_seed(seed=Config.SEED):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Core Model Utilities

## Pytorch Dataset

In [10]:
class ChestXrayDataset(Dataset):
    def __init__(self, df, image_index, target_labels, transform=None, is_xrv=False):
        self.df = df.reset_index(drop=True)
        self.image_index = image_index
        self.target_labels = target_labels
        self.transform = transform
        self.is_xrv = is_xrv
        self.image_names = self.df["Image Index"].values
        self.labels = self.df[self.target_labels].values.astype("float32")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image_path = self.image_index[image_name]

        # Load grayscale image and convert to RGB
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise RuntimeError(f"Failed to load image: {image_path}")

        if not self.is_xrv:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        if self.transform:
            image = self.transform(image)

        label = torch.from_numpy(self.labels[idx])
        return image, label

## Transforms

In [11]:
def get_train_transforms(image_size):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(image_size, scale=(0.9, 1.0)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomRotation(7),
    	transforms.RandomAffine(degrees=5, translate=(0.02, 0.02), scale=(0.98, 1.02)),
        transforms.ColorJitter(brightness=0.05, contrast=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

def get_val_transforms(image_size):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])


In [12]:
def create_dataloaders(
    df,
    image_index,
    target_labels,
    batch_size,
    image_size,
    pin_memory,
    num_workers=4,
    model_arch=""
):

    is_xrv = model_arch == "chexpert_densenet"

    train_transform = get_xrv_train_transforms(image_size) if is_xrv else get_train_transforms(image_size)
    val_transform = get_xrv_val_transforms(image_size) if is_xrv else get_val_transforms(image_size)
    
    train_dataset = ChestXrayDataset(
        df[df["split"] == "train"],
        image_index,
        target_labels,
        transform=train_transform,
        is_xrv=is_xrv
    )
    
    val_dataset = ChestXrayDataset(
        df[df["split"] == "val"],
        image_index,
        target_labels,
        transform=val_transform,
        is_xrv=is_xrv
    )
        
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_memory,
        num_workers=num_workers
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=num_workers
    )
    
    return train_loader, val_loader

## Model Factory

In [13]:
def get_model(name, num_classes=3, pretrained=True):

    if name == "resnet50":

        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

    elif name == "densenet121":

        weights = DenseNet121_Weights.DEFAULT if pretrained else None
        model = densenet121(weights=weights)

        model.classifier = nn.Linear(
            model.classifier.in_features,
            num_classes
        )

    elif name == "efficientnet_b0":

        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = efficientnet_b0(weights=weights)

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes
        )

    elif name == "chexpert_densenet":
        
        model = xrv.models.DenseNet(weights="densenet121-res224-chex")
        model.classifier = nn.Linear(
            model.classifier.in_features,
            num_classes
        )
    
    else:
        raise ValueError("Unknown model")

    return model

## Optimizer Factory

In [14]:
def get_optimizer(name, model, backbone_lr, head_lr, wd):

    head_name = "classifier"
    if isinstance(model, models.ResNet):
        head_name = "fc"

    backbone_params = []
    head_params = []

    for n, p in model.named_parameters():
        if head_name in n:
            head_params.append(p)
        else:
            backbone_params.append(p)

    param_groups = [
        {'params': backbone_params, 'lr': backbone_lr}, # Base learning rate for backbone
        {'params': head_params, 'lr': head_lr} # Faster learning for head
    ]
    
    if name == 'adamw':
        return torch.optim.AdamW(param_groups, weight_decay=wd)
    elif name == 'sgd':
        return torch.optim.SGD(param_groups, momentum=0.9, weight_decay=wd)
    elif name == 'rmsprop':
        return torch.optim.RMSprop(param_groups, weight_decay=wd)

## Focal Loss

In [15]:
class FocalLoss(nn.Module):
    """
    Binary Focal Loss for imbalanced classification.
    Reference: 'Focal Loss for Dense Object Detection' (https://arxiv.org/abs/1708.02002)
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = "mean"):
        """
        Args:
            alpha (float): Weighting factor for the positive class. 
                           Negative class weight is evaluated as (1 - alpha).
                           Set to a negative value to disable alpha weighting.
            gamma (float): Focusing parameter to down-weight easy examples.
            reduction (str): Specifies the reduction to apply to the output:
                             'none' | 'mean' | 'sum'.
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits (torch.Tensor): Raw, unnormalized scores from the model.
            targets (torch.Tensor): Binary targets (0 or 1), same shape as logits.
            
        Returns:
            torch.Tensor: Computed Focal Loss.
        """
        # Ensure targets are floats for BCE
        targets = targets.to(dtype=logits.dtype)
        
        # Calculate standard BCE with logits (numerically stable)
        bce_loss = F.binary_cross_entropy_with_logits(
            logits, 
            targets, 
            reduction="none"
        )
        
        # Calculate pt: probability of the true class
        pt = torch.exp(-bce_loss)
        
        # Standard focal loss formula
        focal_loss = (1 - pt) ** self.gamma * bce_loss
        
        # Apply alpha class-balancing
        if self.alpha >= 0:
            # alpha_t = alpha * y + (1 - alpha) * (1 - y)
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            focal_loss = alpha_t * focal_loss

        # Apply reduction
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

## Model Engine (Train & Validate)

In [16]:
def train_one_epoch(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: typing.Callable,
    device: torch.device,
    scaler: torch.cuda.amp.GradScaler
) -> float:
    
    model.train()
    
    running_loss = 0.0
    total_samples = 0
    
    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type):
            outputs = model(images)
            loss = loss_fn(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size
        
        pbar.set_postfix(loss=loss.item())

    return running_loss / total_samples

In [17]:
def validate(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    loss_fn: typing.Callable,
    device: torch.device
) -> typing.Tuple[float, torch.Tensor, torch.Tensor]:

    model.eval()
    
    running_loss = 0.0
    total_samples = 0
    
    preds = []
    targets = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type):
                outputs = model(images)
                loss = loss_fn(outputs, labels)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            preds.append(outputs.detach().cpu())
            targets.append(labels.detach().cpu())

    preds = torch.cat(preds)
    targets = torch.cat(targets)

    return running_loss / total_samples, preds, targets

In [18]:
def compute_auc(preds: torch.Tensor, targets: torch.Tensor):
    
    preds = torch.sigmoid(preds).detach().cpu().numpy()
    targets = targets.detach().cpu().numpy()


    aucs = roc_auc_score(
        targets,
        preds,
        average=None  
    )
    return aucs

In [19]:
FINAL_CONFIG = {
    "architecture": "densenet121",

    "image_size": 320,
    "batch_size": 32,

    "optimizer": "adamw",

    "backbone_lr": 8.971931734503725e-05,
    "head_lr": 5.903388345924983e-04,

    "weight_decay": 1e-4,

    "alpha": 0.75,
    "gamma": 2.0,

    "epochs": 10,
    "patience": 0,

    "seed": 42
}

In [20]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

wandb.init(
    project="nih-chest-pathologies",
    name="final_densenet121",
    job_type="final_training",
    config=FINAL_CONFIG
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kyrillos_nabil (kyrillosnabil) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [21]:
Config.MODEL_ARCH = FINAL_CONFIG["architecture"]
Config.IMAGE_SIZE = FINAL_CONFIG["image_size"]
Config.BATCH_SIZE = FINAL_CONFIG["batch_size"]
Config.SEED = FINAL_CONFIG["seed"]

set_seed(Config.SEED)

In [22]:
ARTIFACT_DIR = Path("/kaggle/input/datasets/shehabmahmoudhelmy/artifacts")
preprocessed_dataset_path = ARTIFACT_DIR / "prepared_df.pkl"
with open(preprocessed_dataset_path, "rb") as f:
    df = pickle.load(f)

image_index_path = ARTIFACT_DIR /"image_index.pkl"
with open(image_index_path, "rb") as f:
    image_index = pickle.load(f)

In [23]:
train_loader, val_loader = create_dataloaders(
    df=df,
    image_index=image_index,
    target_labels=Config.LABELS,
    batch_size=FINAL_CONFIG["batch_size"],
    image_size=FINAL_CONFIG["image_size"],
    pin_memory=Config.PIN_MEMORY,
    num_workers=Config.NUM_WORKERS,
    model_arch=Config.MODEL_ARCH
)

In [24]:
model = get_model(
    FINAL_CONFIG["architecture"]
).to(Config.DEVICE)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 211MB/s]


In [25]:
optimizer = get_optimizer(
    FINAL_CONFIG["optimizer"],
    model,
    FINAL_CONFIG["backbone_lr"],
    FINAL_CONFIG["head_lr"],
    FINAL_CONFIG["weight_decay"]
)

In [26]:
loss_fn = FocalLoss(
    alpha=FINAL_CONFIG["alpha"],
    gamma=FINAL_CONFIG["gamma"]
)

In [27]:
scaler = torch.amp.GradScaler(Config.DEVICE)

best_auc = -1.0
best_epoch = -1

checkpoint_path = (
    Path(Config.MODEL_DIR)
    / "final_densenet121_best.pt"
)

In [28]:
for epoch in range(Config.EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        loss_fn,
        Config.DEVICE,
        scaler
    )

    val_loss, preds, targets = validate(
        model,
        val_loader,
        loss_fn,
        Config.DEVICE
    )

    auc_per_class = compute_auc(
        preds,
        targets
    )

    mean_auc = auc_per_class.mean()

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_mean_auc": mean_auc,

        "atelectasis_auc": auc_per_class[0],
        "effusion_auc": auc_per_class[1],
        "pneumothorax_auc": auc_per_class[2],

        "backbone_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f} | "
        f"Mean AUC: {mean_auc:.5f}"
    )

    if mean_auc > best_auc:

        best_auc = mean_auc
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_auc": best_auc,
                "config": FINAL_CONFIG
            },
            checkpoint_path
        )

        print(
            f"New best model saved "
            f"(Epoch {epoch + 1}, AUC={best_auc:.5f})"
        )

wandb.summary["best_auc"] = float(best_auc)
wandb.summary["best_epoch"] = int(best_epoch)

print("\n" + "=" * 60)
print("FINAL TRAINING COMPLETE")
print("=" * 60)

print(f"Best Epoch     : {best_epoch + 1}")
print(f"Best Mean AUC  : {best_auc:.5f}")
print(f"Checkpoint     : {checkpoint_path}")

wandb.finish()

Epoch 00 | Train Loss: 0.03591 | Val Loss: 0.03381 | Mean AUC: 0.85532
New best model saved (Epoch 1, AUC=0.85532)


Epoch 01 | Train Loss: 0.03331 | Val Loss: 0.03590 | Mean AUC: 0.85841
New best model saved (Epoch 2, AUC=0.85841)


Epoch 02 | Train Loss: 0.03200 | Val Loss: 0.03470 | Mean AUC: 0.85997
New best model saved (Epoch 3, AUC=0.85997)


Epoch 03 | Train Loss: 0.03118 | Val Loss: 0.03361 | Mean AUC: 0.87163
New best model saved (Epoch 4, AUC=0.87163)


Epoch 04 | Train Loss: 0.03044 | Val Loss: 0.03179 | Mean AUC: 0.86642


...

Epoch 26 | Train Loss: 0.01354 | Val Loss: 0.06610 | Mean AUC: 0.84294


Epoch 27 | Train Loss: 0.01282 | Val Loss: 0.07700 | Mean AUC: 0.84673


Epoch 28 | Train Loss: 0.01244 | Val Loss: 0.07802 | Mean AUC: 0.84282


Epoch 29 | Train Loss: 0.01240 | Val Loss: 0.07190 | Mean AUC: 0.84227

FINAL TRAINING COMPLETE
Best Epoch     : 9
Best Mean AUC  : 0.87466
Checkpoint     : models/final_densenet121_best.pt


atelectasis_auc,▃▅▅█▇█▇▇█▇▇▆▆▆▆▅▁▃▄▃▄▃▃▃▂▂▃▂▂▂
backbone_lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
effusion_auc,▇▇▆█▇████▇█▆▇▇▅▇▆▅▆▅▅▁▅▃▅▅▂▄▂▄
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
head_lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
pneumothorax_auc,▅▃▅▆▅▅▃▆█▆▇▅▃▅▅▅▄▂▅▁▃▁▃▄▂▅▃▄▅▂
train_loss,█▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁
val_loss,▁▂▁▁▁▁▁▁▁▂▁▂▂▂▂▄▃▄▃▄▅▄▅▅▆▇▆██▇
val_mean_auc,▄▅▅▇▆▇▆▇█▇▇▅▅▆▅▆▃▃▄▃▄▁▃▂▂▃▂▃▂▂
atelectasis_auc,0.78116
backbone_lr,9e-05
